# Chapter 16 &mdash; Corralling Problems: NP, NPC, and the Collapse Property

**Concept 2 of the Chapter 16 decomposition:** *Corralling Problems: NP, NPC, and the Collapse Property*

Group the checkable problems into NP, identify the hardest as NPC &mdash; one polynomial NPC algorithm collapses everything.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-Corralling-Problems/Concept-Corralling-Problems.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


The organising move of the subject: rather than study thousands of hard problems one
at a time, **corral** them.

* **NP** &mdash; problems with polynomial-time-checkable certificates.
* **NP-hard** &mdash; problems to which *every* NP problem reduces in polynomial time.
* **NP-complete (NPC)** &mdash; in NP **and** NP-hard: the hardest problems *in* NP.

The **collapse property** is what makes this worth doing:

> If **any** NPC problem has a polynomial algorithm, then **every** NP problem does,
> and $P = NP$.

So thousands of independent research efforts became one question. And the practical
reading of "your problem is NPC" is: *nobody will find a fast exact algorithm for it
without solving the biggest open problem in the field.*

## 2. Definitions

### The corral, as data

In [ ]:
CLASSES = [
 ("P",       "solvable in polynomial time",                  ["2-SAT", "shortest path", "primality"]),
 ("NP",      "certificate checkable in polynomial time",     ["SAT", "clique", "TSP-decision", "all of P"]),
 ("NP-hard", "every NP problem reduces to it",               ["SAT", "clique", "halting problem"]),
 ("NPC",     "in NP AND NP-hard",                            ["3-SAT", "clique", "vertex cover", "TSP-decision"]),
]

REDUCTIONS = [("3-SAT", "clique"), ("clique", "vertex cover"),
              ("3-SAT", "subset sum"), ("subset sum", "partition"),
              ("SAT", "3-SAT")]

### The collapse, simulated

In [ ]:
def collapse(known_poly, reductions, npc):
    # if `known_poly` has a polynomial algorithm, what else follows?
    solved, frontier = {known_poly}, {known_poly}
    while frontier:
        nxt = {a for a, b in reductions if b in solved} | \
              {b for a, b in reductions if a in solved}
        nxt -= solved
        solved |= nxt; frontier = nxt
    return solved

<!-- nav-strip -->

---

&larr;&nbsp;[Ch16&nbsp;1.&nbsp;Hard to Solve, Easy to Check: TSP and the Idea of a Certificate](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-Hard-To-Solve-Easy-To-Check/Concept-Hard-To-Solve-Easy-To-Check.ipynb) &nbsp;&middot;&nbsp; [**Chapter 16** index](https://github.com/ganeshutah/Jove/blob/master/Chapter16/README.md) &nbsp;&middot;&nbsp; [Ch16&nbsp;3.&nbsp;$P$-time and NP-time Defined via DTMs and NDTMs](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-P-And-NP-Via-Machines/Concept-P-And-NP-Via-Machines.ipynb)&nbsp;&rarr;

---

## 3. Tests

The classes, and who lives where.

In [ ]:
for name, what, members in CLASSES:
    print("%-9s %-44s %s" % (name, what, ', '.join(members)))

Note that **NP-hard is not a subset of NP**.

In [ ]:
print("the halting problem is NP-hard (everything reduces to it)")
print("   ... and not in NP: it is not even decidable")
print()
print("NPC = NP-hard INTERSECT NP.  Both halves are load-bearing.")
print("(Concept 10 returns to this with Diophantine equations.)")

The **collapse property**, traced through the reduction graph.

In [ ]:
reach = collapse("3-SAT", REDUCTIONS, None)
print("if 3-SAT became polynomial, these follow :", sorted(reach))
assert "clique" in reach and "vertex cover" in reach

Which is why one algorithm would settle the whole class.

In [ ]:
print("every NP problem  <=p  3-SAT        (Cook-Levin, Concept 8)")
print("3-SAT in P                          (hypothetically)")
print("=> every NP problem in P            (compose the two)")
print("=> P = NP")

And the contrapositive, which is how the result is actually used.

In [ ]:
print("Your problem is NPC.  Therefore:")
print("  * no polynomial exact algorithm is known, and finding one would be")
print("    a historic result -- do not budget for it;")
print("  * spend the effort on approximation, heuristics, special cases,")
print("    or a SAT solver (Concept 12) instead.")

## 4. Exercises


1. Why is $P \subseteq NP$ immediate?
2. If some NPC problem were proved to need exponential time, what follows?
3. Name a problem in NP that is not known to be in P or known to be NPC.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter16/Concept-Corralling-Problems')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')